<div style="padding:10px; 
            color:#FF9F00;
            margin:10px;
            font-size:150%;
            display:fill;
            border-radius:1px;
            border-style: solid;
            border-color:#FF9F00;
            background-color:#3E3D53;
            overflow:hidden;">
    <center>
        <a id='top'></a>
        <b>Table of Contents</b>
    </center>
    <br>
    <ul>
        <li>
            <a href="#1">1 -  Overview and Imports</a>
        </li>
        <li>
            <a href="#2">2 - Data Preparation</a>
        </li>
        <li>
            <a href="#3">3 - LTC Implementation</a>
        <li>
            <a href="#4">4 - Thank you</a>
        </li> 
    </ul>
</div>
<a id="1"></a>

<h1 style='background:#FF9F00;border:0; color:black;
    box-shadow: 10px 10px 5px 0px rgba(0,0,0,0.75);
    transform: rotateX(10deg);
    '><center style='color: #3E3D53;'>Overview and Imports</center></h1>
    
# Overview and Imports

**Liquid Time-Constant (LTC) networks are a type of neural network inspired by biological neurons that are designed to handle sequential data, such as time series or natural language text. They achieve this by using differential equations to model the continuous dynamics of neurons, allowing the network to maintain a memory of previous inputs over time with adaptive time constants.**

**This notebook contains an implementation of an LTC network that can be used for language modeling. The model takes in a sequence of characters and outputs the probability distribution over the next character in the sequence. The network is trained on a corpus of text and then used to generate new text that has a similar distribution of characters as the training corpus.**

In [30]:
import os
import numpy as np
import scipy as sp


In [31]:
class LiquidTimeConstantCell:
    """
    Pure NumPy implementation of Liquid Time-Constant (LTC) Neural Network Cell
    Based on the paper: https://arxiv.org/abs/1905.12374
    
    This implementation uses only numpy and scipy, avoiding PyTorch dependencies.
    """
    
    def __init__(self, num_units, input_size=None):
        self.num_units = num_units
        self.input_size = input_size
        self.is_built = False
        
        # Hyperparameters
        self.ode_solver_unfolds = 6
        self.erev_init_factor = 1
        self.w_init_max = 1.0
        self.w_init_min = 0.01
        self.cm_init_min = 0.5
        self.cm_init_max = 0.5
        self.gleak_init_min = 1.0
        self.gleak_init_max = 1.0
        
        # Parameter bounds
        self.w_min_value = 0.00001
        self.w_max_value = 1000
        self.gleak_min_value = 0.00001
        self.gleak_max_value = 1000
        self.cm_t_min_value = 0.000001
        self.cm_t_max_value = 1000
        
    def build(self, input_size):
        """Initialize all parameters"""
        self.input_size = input_size
        
        # Input mapping parameters (Affine mapping)
        self.input_w = np.ones(input_size, dtype=np.float64)
        self.input_b = np.zeros(input_size, dtype=np.float64)
        
        # Sensory (input) parameters
        self.sensory_mu = (np.random.rand(input_size, self.num_units) * 0.5 + 0.3).astype(np.float64)
        self.sensory_sigma = (np.random.rand(input_size, self.num_units) * 5.0 + 3.0).astype(np.float64)
        self.sensory_W = np.random.uniform(
            low=self.w_init_min, high=self.w_init_max, 
            size=(input_size, self.num_units)
        ).astype(np.float64)
        sensory_erev_init = (2 * np.random.randint(0, 2, size=(input_size, self.num_units)) - 1).astype(np.float64)
        self.sensory_erev = (sensory_erev_init * self.erev_init_factor).astype(np.float64)
        
        # Recurrent parameters
        self.mu = (np.random.rand(self.num_units, self.num_units) * 0.5 + 0.3).astype(np.float64)
        self.sigma = (np.random.rand(self.num_units, self.num_units) * 5.0 + 3.0).astype(np.float64)
        self.W = np.random.uniform(
            low=self.w_init_min, high=self.w_init_max,
            size=(self.num_units, self.num_units)
        ).astype(np.float64)
        erev_init = (2 * np.random.randint(0, 2, size=(self.num_units, self.num_units)) - 1).astype(np.float64)
        self.erev = (erev_init * self.erev_init_factor).astype(np.float64)
        
        # Neuron parameters
        self.vleak = (np.random.rand(self.num_units) * 0.4 - 0.2).astype(np.float64)
        if self.gleak_init_max > self.gleak_init_min:
            self.gleak = (np.random.rand(self.num_units) * 
                         (self.gleak_init_max - self.gleak_init_min) + 
                         self.gleak_init_min).astype(np.float64)
        else:
            self.gleak = np.full(self.num_units, self.gleak_init_min, dtype=np.float64)
            
        if self.cm_init_max > self.cm_init_min:
            self.cm_t = (np.random.rand(self.num_units) * 
                        (self.cm_init_max - self.cm_init_min) + 
                        self.cm_init_min).astype(np.float64)
        else:
            self.cm_t = np.full(self.num_units, self.cm_init_min, dtype=np.float64)
            
        self.is_built = True
        
    def _sigmoid(self, v_pre, mu, sigma):
        """Calculate sigmoid activation with mu and sigma parameters (following official implementation)"""
        # Handle 1D input case
        if len(v_pre.shape) == 1:
            v_pre = v_pre.reshape(1, -1)  # Add batch dimension: (1, num_units)
            
        # For proper broadcasting with different parameter shapes
        if len(mu.shape) == 2:  # (input_size, num_units) or (num_units, num_units)
            if mu.shape[0] == v_pre.shape[1]:  # input case: mu is (input_size, num_units)
                v_pre_expanded = v_pre[:, :, np.newaxis]  # (1, input_size, 1)
            else:  # recurrent case: mu is (num_units, num_units)  
                v_pre_expanded = v_pre[:, np.newaxis, :]  # (1, 1, num_units)
        else:
            v_pre_expanded = v_pre
            
        # Compute sigmoid: σ(sigma * (v_pre - mu))
        mues = v_pre_expanded - mu
        x = sigma * mues
        sigmoid_result = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        
        return sigmoid_result.squeeze(0) if sigmoid_result.shape[0] == 1 else sigmoid_result
    
    def _constrain_parameters(self):
        """Constrain parameters to their specified bounds (following official implementation)"""
        self.cm_t = np.clip(self.cm_t, self.cm_t_min_value, self.cm_t_max_value)
        self.gleak = np.clip(self.gleak, self.gleak_min_value, self.gleak_max_value)
        self.W = np.clip(self.W, self.w_min_value, self.w_max_value)
        self.sensory_W = np.clip(self.sensory_W, self.w_min_value, self.w_max_value)
 
    def forward(self, inputs, state):
        """
        Correct LTC forward pass following official implementation - processes ONE time step
        
        Args:
            inputs: numpy array of shape (input_size,) - single input vector
            state: numpy array of shape (num_units,) - current state
            
        Returns:
            outputs: numpy array of shape (num_units,) - new state
            new_state: numpy array of shape (num_units,) - same as outputs
        """
        if not self.is_built:
            input_size = inputs.shape[0] if len(inputs.shape) == 1 else inputs.shape[1]
            self.build(input_size)
            
        # Ensure 1D arrays for single time step processing
        if len(inputs.shape) > 1:
            inputs = inputs.flatten()
        if len(state.shape) > 1:
            state = state.flatten()
            
        # Map inputs (affine transformation)
        mapped_inputs = inputs * self.input_w + self.input_b
        
        # Current state
        v_pre = state.copy().astype(np.float64)
        
        # Compute sensory activations (constant for all ODE steps)
        sensory_sigmoid = self._sigmoid(mapped_inputs, self.sensory_mu, self.sensory_sigma)
        sensory_w_activation = self.sensory_W * sensory_sigmoid
        sensory_rev_activation = sensory_w_activation * self.sensory_erev
        
        # Sum over input dimension, result shape: (num_units,)
        w_numerator_sensory = np.sum(sensory_rev_activation, axis=0)  # Sum over input_size
        w_denominator_sensory = np.sum(sensory_w_activation, axis=0)  # Sum over input_size
        
        # Semi-implicit Euler method (6 steps as in official implementation)
        for t in range(self.ode_solver_unfolds):
            # Compute recurrent activations
            recurrent_sigmoid = self._sigmoid(v_pre, self.mu, self.sigma)
            w_activation = self.W * recurrent_sigmoid
            rev_activation = w_activation * self.erev
            
            # Sum over recurrent dimension, result shape: (num_units,)
            w_numerator = np.sum(rev_activation, axis=0) + w_numerator_sensory
            w_denominator = np.sum(w_activation, axis=0) + w_denominator_sensory
            
            # Semi-implicit Euler update (CRITICAL LTC formula)
            numerator = self.cm_t * v_pre + self.gleak * self.vleak + w_numerator
            denominator = self.cm_t + self.gleak + w_denominator
            
            v_pre = numerator / denominator
            
        # Constrain parameters to bounds
        self._constrain_parameters()
        
        return v_pre, v_pre
        
    def get_parameters(self):
        """Get all trainable parameters as a list of numpy arrays"""
        if not self.is_built:
            return []
            
        return [
            self.input_w, self.input_b,
            self.sensory_mu, self.sensory_sigma, self.sensory_W, self.sensory_erev,
            self.mu, self.sigma, self.W, self.erev,
            self.vleak, self.gleak, self.cm_t
        ]
        
    def set_parameters(self, params):
        """Set all trainable parameters from a list of numpy arrays"""
        if not self.is_built or len(params) != 13:
            raise ValueError("Parameters not compatible with current model state")
            
        # Ensure all parameters are float64
        params = [p.astype(np.float64) for p in params]
        
        (self.input_w, self.input_b,
         self.sensory_mu, self.sensory_sigma, self.sensory_W, self.sensory_erev,
         self.mu, self.sigma, self.W, self.erev,
         self.vleak, self.gleak, self.cm_t) = params


In [32]:
class DataGenerator:
    """
    A class for reading and preprocessing text data.
    """

    def __init__(self, path: str, sequence_length: int):
        """
        Initializes a DataReader object with the path to a text file and the desired sequence length.

        Args:
            path (str): The path to the text file.
            sequence_length (int): The length of the sequences that will be fed to the self.
        """
        with open(path) as f:
            # Read the contents of the file
            self.data = f.read()

        # Find all unique characters in the text
        chars = list(set(self.data))

        # Create dictionaries to map characters to indices and vice versa
        self.char_to_idx = {ch: i for (i, ch) in enumerate(chars)}
        self.idx_to_char = {i: ch for (i, ch) in enumerate(chars)}

        # Store the size of the text data and the size of the vocabulary
        self.data_size = len(self.data)
        self.vocab_size = len(chars)

        # Initialize the pointer that will be used to generate sequences
        self.pointer = 0

        # Store the desired sequence length
        self.sequence_length = sequence_length


    def next_batch(self):
        """
        Generates a batch of input and target sequences.

        Returns:
            inputs_one_hot (np.ndarray): A numpy array with shape `(batch_size, vocab_size)` where each row is a one-hot encoded representation of a character in the input sequence.
            targets (list): A list of integers that correspond to the indices of the characters in the target sequence, which is the same as the input sequence shifted by one position to the right.
        """
        input_start = self.pointer
        input_end = self.pointer + self.sequence_length

        # Get the input sequence as a list of integers
        inputs = [self.char_to_idx[ch] for ch in self.data[input_start:input_end]]

        # One-hot encode the input sequence
        inputs_one_hot = np.zeros((len(inputs), self.vocab_size))
        # print('batch_size:', (len(inputs)))
        inputs_one_hot[np.arange(len(inputs)), inputs] = 1

        # Get the target sequence as a list of integers
        targets = [self.char_to_idx[ch] for ch in self.data[input_start + 1:input_end + 1]]

        # Update the pointer
        self.pointer += self.sequence_length

        # Reset the pointer if the next batch would exceed the length of the text data
        if self.pointer + self.sequence_length + 1 >= self.data_size:
            self.pointer = 0

        return inputs_one_hot, targets

In [33]:
def cut_in_sequences(x,y,seq_len,inc=1):

    sequences_x = []
    sequences_y = []

    for s in range(0,x.shape[0] - seq_len,inc):
        start = s
        end = start+seq_len
        sequences_x.append(x[start:end])
        sequences_y.append(y[start:end])

    return np.stack(sequences_x,axis=1),np.stack(sequences_y,axis=1)


numID = 13


class HarData:

    def __init__(self,seq_len=16):
        print("Parsing for Patient File {}".format(numID))

        train_x = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_glucoseV2.txt")
        train_y = (np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_glucoseV2.txt") - 1)  # .astype(np.int32)
        train_ins = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_bolus.txt")
        train_meal = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_meal.txt")
        train_basal = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_basal.txt")
        train_initIsc1 = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Isc1.txt")
        train_initIsc2 = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Isc2.txt")
        train_initIp = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Ip.txt")
        train_BW = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_BW.txt")
        train_u2ss = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_u2ss.txt")

        

        train_meal2 = train_meal

        test_x = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_glucoseV2.txt")
        test_y = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_glucoseV2.txt") - 1  # .astype(np.int32)
        test_ins = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_bolus.txt")
        test_meal = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_meal.txt")
        test_meal2 = test_meal
        test_basal = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_basal.txt")
        test_initIsc1 = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Isc1.txt")
        test_initIsc2 = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Isc2.txt")
        test_initIp = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_Ip.txt")
        test_BW = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_BW.txt")
        test_u2ss = np.loadtxt(f"ohioT1DAyan-main\\Virtual_570_{numID}_u2ss.txt")

        
        # shape = tf.shape(test_basal)
        # with tf.compat.v1.Session() as sess:
        #     numpy_number = sess.run(shape[1])
        # Nloop = numpy_number
        # print("Nloop {}".format(Nloop))
        train_x,train_y = cut_in_sequences(train_x,train_y,seq_len)
        train_ins,train_meal = cut_in_sequences(train_ins,train_meal,seq_len)
        train_basal,train_initIsc1 = cut_in_sequences(train_basal,train_initIsc1,seq_len)
        train_initIsc2, train_initIp = cut_in_sequences(train_initIsc2, train_initIp, seq_len)
        train_BW, train_u2ss = cut_in_sequences(train_BW, train_u2ss, seq_len)

        test_x,test_y = cut_in_sequences(test_x,test_y,seq_len,inc=8)
        test_ins,test_meal = cut_in_sequences(test_ins,test_meal,seq_len,inc=8)

        test_basal, test_initIsc1 = cut_in_sequences(test_basal, test_initIsc1, seq_len,inc=8)
        test_initIsc2, test_initIp = cut_in_sequences(test_initIsc2, test_initIp, seq_len,inc=8)
        test_BW, test_u2ss = cut_in_sequences(test_BW, test_u2ss, seq_len,inc=8)
        print("Total number of testing sequences: {}".format(test_initIsc1.shape[1]))
        #permutation = np.random.RandomState(893429).permutation(train_x.shape[1])
        valid_size = int(0.1*train_x.shape[1])
        print("Validation split: {}, training split: {}".format(valid_size,train_x.shape[1]-valid_size))

        self.valid_x = train_x[:,:valid_size]
        self.valid_ins = train_ins[:,:valid_size]
        self.valid_meal = train_meal[:,:valid_size]
        self.valid_y = train_y[:,:valid_size]
        self.valid_basal = train_basal[:,:valid_size]
        self.valid_initIsc1 = train_initIsc1[:,:valid_size]
        self.valid_initIsc2 = train_initIsc2[:, :valid_size]
        self.valid_initIp = train_initIp[:, :valid_size]
        self.valid_BW = train_BW[:, :valid_size]
        self.valid_u2ss = train_u2ss[:, :valid_size]



        self.train_x = train_x[:,valid_size:]
        self.train_ins = train_ins[:,valid_size:]
        self.train_meal = train_meal[:,valid_size:]
        self.train_y = train_y[:,valid_size:]
        self.train_basal = train_basal[:,valid_size:]
        self.train_initIsc1 = train_initIsc1[:, valid_size:]
        self.train_initIsc2 = train_initIsc2[:, valid_size:]
        self.train_initIp = train_initIp[:, valid_size:]
        self.train_BW = train_BW[:, valid_size:]
        self.train_u2ss = train_u2ss[:, valid_size:]

        self.test_x = test_x
        self.test_ins = test_ins
        self.test_meal = test_meal
        self.test_y = test_y
        self.test_basal = test_basal
        self.test_initIsc1 = test_initIsc1
        self.test_initIsc2 = test_initIsc2
        self.test_initIp = test_initIp
        self.test_BW = test_BW
        self.test_u2ss = test_u2ss


        
        # print("train_x: {}".format(self.train_x.shape))
        # print("train_y: {}".format(self.train_y.shape))
        # print("train_ins: {}".format(self.train_ins.shape))
        # print("train_meal: {}".format(self.train_meal.shape))
        # print("train_basal: {}".format(self.train_basal.shape))
        # print("train_initIsc1: {}".format(train_initIsc1.shape))
        # print("train_initIsc2: {}".format(train_initIsc2.shape))
        # print("train_initIp: {}".format(train_initIp.shape))
        # print("train_BW: {}".format(train_BW.shape))
        # print("train_u2ss: {}".format(train_u2ss.shape))

        # print("test_x: {}".format(test_x.shape))
        # print("test_y: {}".format(test_y.shape))
        # print("test_ins: {}".format(test_ins.shape))
        # print("test_meal: {}".format(test_meal.shape))
        # print("test_basal: {}".format(test_basal.shape))
        # print("test_initIsc1: {}".format(test_initIsc1.shape))
        # print("test_initIsc2: {}".format(test_initIsc2.shape))
        # print("test_initIp: {}".format(test_initIp.shape))
        # print("test_BW: {}".format(test_BW.shape))
        # print("test_u2ss: {}".format(test_u2ss.shape))


        print("Total number of test sequences: {}".format(self.test_x.shape[1]))

In [34]:
class LTC:
    """
    A class used to represent a Liquid Time-Constant (LTC) Neural Network.

    Attributes
    ----------
    hidden_size : int
        The number of hidden units in the LTC.
    vocab_size : int
        The size of the vocabulary used by the LTC.
    sequence_length : int
        The length of the input sequences fed to the LTC.
    learning_rate : float
        The learning rate used during training.

    Methods
    -------
    __init__(hidden_size, vocab_size, sequence_length, learning_rate)
        Initializes an instance of the LTC class.
    """

    def __init__(self, hidden_size, vocab_size, sequence_length, learning_rate):
        """
        Initializes an instance of the LTC class.

        Parameters
        ----------
        hidden_size : int
            The number of hidden units in the LTC.
        vocab_size : int
            The size of the vocabulary used by the LTC.
        sequence_length : int
            The length of the input sequences fed to the LTC.
        learning_rate : float
            The learning rate used during training.
        """
        # hyper parameters
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.sequence_length = sequence_length
        self.learning_rate = learning_rate
        
        # Training control
        self.train_ltc_params = False  # Start by only training output layer
        self.iteration_count = 0

        # Initialize LiquidNet (LTC) core with pure numpy implementation
        self.ltc_cell = LiquidTimeConstantCell(hidden_size)
        
        # Output layer parameters
        self.Wy = np.random.uniform(-np.sqrt(1. / hidden_size), np.sqrt(1. / hidden_size),
                                    (vocab_size, hidden_size)).astype(np.float64)
        self.by = np.zeros((vocab_size, 1), dtype=np.float64)

        # initialize gradients for all parameters (LTC + output layer)
        self.dWy = np.zeros_like(self.Wy, dtype=np.float64)
        self.dby = np.zeros_like(self.by, dtype=np.float64)
        self.ltc_gradients = []  # Will store gradients for LTC parameters

        # initialize parameters for adamw optimizer
        self.mWy = np.zeros_like(self.Wy, dtype=np.float64)
        self.vWy = np.zeros_like(self.Wy, dtype=np.float64)
        self.mby = np.zeros_like(self.by, dtype=np.float64)
        self.vby = np.zeros_like(self.by, dtype=np.float64)
        self.ltc_m_params = []  # AdamW momentum for LTC parameters
        self.ltc_v_params = []  # AdamW velocity for LTC parameters

    def sigmoid(self, x):
        """
        Computes the sigmoid activation function for a given input array.

        Parameters:
            x (ndarray): Input array.

        Returns:
            ndarray: Array of the same shape as `x`, containing the sigmoid activation values.
        """
        return 1 / (1 + np.exp(-x))

    def softmax(self, x):
        """
        Computes the softmax activation function for a given input array.

        Parameters:
            x (ndarray): Input array.

        Returns:
            ndarray: Array of the same shape as `x`, containing the softmax activation values.
        """
        # shift the input to prevent overflow when computing the exponentials
        x = x - np.max(x)
        # compute the exponentials of the shifted input
        p = np.exp(x)
        # normalize the exponentials by dividing by their sum
        return p / np.sum(p)

    def forward(self, X, c_prev, a_prev):
        """
        Performs forward propagation for a Liquid Time-Constant (LTC) model.

        Args:
            X (numpy array): Input sequence, shape (sequence_length, input_size)
            c_prev (numpy array): Previous cell state, shape (hidden_size, 1) - unused for LTC
            a_prev (numpy array): Previous hidden state, shape (hidden_size, 1)

        Returns: X (numpy array): Input sequence, shape (sequence_length, input_size) 
        r, z, c, cc (dictionary): Placeholder dictionaries for compatibility
        a (dictionary): Hidden state for each time step, keys = time step, values = numpy array shape (hidden_size, 1) 
        y_pred (dictionary): Output probability vector for each time step, keys = time step, values = numpy array shape (output_size, 1)
        """
        # Initialize dictionaries for compatibility with existing code
        r, z, c, cc, a, y_pred = {}, {}, {}, {}, {}, {}
        
        # Initialize state
        state = a_prev.flatten()  # LTC expects 1D state
        
        # Store initial state
        a[-1] = np.copy(a_prev)
        c[-1] = np.copy(c_prev)  # Keep for compatibility
        
        # Iterate over each time step in the input sequence
        for t in range(X.shape[0]):
            # Get current input
            input_t = X[t, :]  # Shape: (input_size,)
            
            # Forward pass through LTC cell
            ltc_output, state = self.ltc_cell.forward(input_t, state)
            
            # Reshape for compatibility
            hidden_state = ltc_output.reshape(-1, 1)  # Shape: (hidden_size, 1)
            
            # Store states (r, z, cc are placeholders for compatibility)
            r[t] = np.zeros_like(hidden_state)
            z[t] = np.zeros_like(hidden_state)
            c[t] = np.copy(hidden_state)
            cc[t] = np.zeros_like(hidden_state)
            a[t] = np.copy(hidden_state)

            # Compute the output probability vector using the output layer
            y_pred[t] = self.softmax(np.dot(self.Wy, a[t]) + self.by)

        # Return the output probability vectors and states
        return X, r, z, c, cc, a, y_pred

    def backward(self, X, a_prev, c_prev, r, z, c, cc, a, y_pred, targets):
        """
        Performs backward propagation through time for an LTC network.

        Args:
            X (numpy array): Input sequence, shape (sequence_length, input_size)
            a_prev (numpy array): Previous hidden state, shape (hidden_size, 1)
            r, z, c, cc (dictionary): Placeholder dictionaries for compatibility
            a (dictionary): Hidden state for each time step, keys = time step, values = numpy array shape (hidden_size, 1)
            y_pred (dictionary): Output probability vector for each time step, keys = time step, values = numpy array shape (output_size, 1)
            targets (numpy array): Target outputs for each time step, shape (sequence_length, output_size)

        Returns:
            None       
        """
        # Reset gradients for output layer
        self.dWy.fill(0)
        self.dby.fill(0)
        
        # Initialize gradients for LTC parameters if not done yet
        if not self.ltc_gradients:
            ltc_params = self.ltc_cell.get_parameters()
            self.ltc_gradients = [np.zeros_like(param, dtype=np.float64) for param in ltc_params]
        else:
            # Reset LTC gradients
            for grad in self.ltc_gradients:
                grad.fill(0)
        
        # Iterate through time steps to compute gradients
        for t in range(X.shape[0]):
            # Compute the gradient of the output probability vector
            dy = np.copy(y_pred[t])
            dy[targets[t]] -= 1

            # Compute the gradient of the output layer weights and biases
            self.dWy += np.dot(dy, a[t].T)
            self.dby += dy
            
            # Compute gradient w.r.t. hidden state
            da = np.dot(self.Wy.T, dy)
            
            # Simplified LTC gradient computation
            # We'll use the gradient w.r.t. hidden states to approximate LTC parameter gradients
            if self.ltc_cell.is_built:
                ltc_params = self.ltc_cell.get_parameters()
                
                # Get the current input for this timestep
                input_t = X[t, :]
                
                # Compute approximate gradients for LTC parameters
                # This is a simplified approximation - proper LTC gradients require
                # backpropagation through the ODE solver
                
                gradient_scale = np.mean(np.abs(da)) * 0.01  # Scale down the gradients
                
                for i, param in enumerate(ltc_params):
                    if i < 2:  # input_w, input_b (input mapping)
                        if param.shape == input_t.shape:
                            self.ltc_gradients[i] += (da.flatten()[:param.size].reshape(param.shape) * gradient_scale).astype(np.float64)
                        else:
                            self.ltc_gradients[i] += (np.ones_like(param) * gradient_scale).astype(np.float64)
                    elif i < 6:  # sensory parameters
                        # Use a small fraction of the hidden state gradient
                        self.ltc_gradients[i] += (np.ones_like(param) * gradient_scale * 0.1).astype(np.float64)
                    else:  # recurrent and neuron parameters
                        # Use an even smaller fraction for stability
                        self.ltc_gradients[i] += (np.ones_like(param) * gradient_scale * 0.01).astype(np.float64)

        # Clip gradients to avoid exploding gradients
        np.clip(self.dWy, -1, 1, out=self.dWy)
        np.clip(self.dby, -1, 1, out=self.dby)
        
        # Clip LTC gradients
        for grad in self.ltc_gradients:
            np.clip(grad, -1, 1, out=grad)

    # def loss(self, y_preds, targets):
    #     """
    #     Computes the cross-entropy loss for a given sequence of predicted probabilities and true targets.

    #     Parameters:
    #         y_preds (ndarray): Array of shape (sequence_length, vocab_size) containing the predicted probabilities for each time step.
    #         targets (ndarray): Array of shape (sequence_length, 1) containing the true targets for each time step.

    #     Returns:
    #         float: Cross-entropy loss.
    #     """
    #     # calculate cross-entropy loss

    #     result = sum(-np.log(y_preds[t][targets[t], 0]) for t in range(self.sequence_length))
    #     print('result:', result.shape)


    #     return sum(-np.log(y_preds[t][targets[t], 0]) for t in range(self.sequence_length))

    
    def loss(self, y_preds, targets, y_initIsc1, y_initIsc2, y_initIp, train_u2ss, train_basal, train_ins, train_meal):
        # har_data_instance = HarData()  # Create an instance of HarData

        # # Convert 3D array self.y_initIsc1 to 2D by taking the first slice along the first axis
        # self.y_initIsc1 = har_data_instance.train_initIsc1[0, :, :]
        # # print('y_initIsc1 (converted to 2D):', self.y_initIsc1.shape)

        # # Convert 3D array self.y_initIsc2 to 2D by taking the first slice along the first axis
        # self.y_initIsc2 = har_data_instance.train_initIsc2[0, :, :]
        # # print('y_initIsc2 (converted to 2D):', self.y_initIsc2.shape)

        # # Convert 3D array self.y_initIp to 2D by taking the first slice along the first axis
        # self.y_initIp = har_data_instance.train_initIp[0, :, :]
        # # print('y_initIp (converted to 2D):', self.y_initIp.shape)

        # # Convert 3D array self.train_u2ss to 2D by taking the first slice along the first axis
        # self.train_u2ss = har_data_instance.train_u2ss[0, :, :]
        # # print('train_u2ss (converted to 2D):', self.train_u2ss.shape)

        # # Convert 3D array self.train_basal to 2D by taking the first slice along the first axis
        # self.train_basal = har_data_instance.train_basal[0, :, :]
        # # print('train_basal (converted to 2D):', self.train_basal.shape)

        # # Convert 3D array self.train_ins to 2D by taking the first slice along the first axis
        # self.train_ins = har_data_instance.train_ins[0, :, :]
        # # print('train_ins (converted to 2D):', self.train_ins.shape)

        # # Convert 3D array self.train_meal to 2D by taking the first slice along the first axis
        # self.train_meal = har_data_instance.train_meal[0, :, :]
        # # print('train_meal (converted to 2D):', self.train_meal.shape)
        



        ############################# Parameters ##################################
        maxChange = 50
        kempt = (1 + (0.5 - y_preds[0]) * maxChange / 100) * 0.18
        kabs = (1 + (0.5 - y_preds[1]) * maxChange / 100) * 0.012
        f = 0.9
        Gb = (1 + (0.5 - y_preds[ 2]) * maxChange / 100) * 119.13
        SG = (1 + (0.5 - y_preds[3]) * maxChange / 100) * 0.025
        Vg = 1.45
        p2 = (1 + (0.5 - y_preds[4]) * maxChange / 100) * 0.012
        SI = (1 + (0.5 - y_preds[5]) * maxChange / 100) * 0.001035 / Vg
        Ipb = (1 + (0.5 - y_preds[6]) * maxChange / 100)
        alpha = 7
        kd = (1 + (0.5 - y_preds[7]) * maxChange / 100) * 0.026
        beta = np.floor((1 + (0.5 - y_preds[9]) * maxChange / 100) * 15)
        Vi = 0.126
        ka2 = (1 + (0.5 - y_preds[8]) * maxChange / 100) * 0.014
        ke = 0.127
        bolusD = np.floor((1 + (0.5 - y_preds[10]) * maxChange / 100) * 5)
        r2 = 0.8124

        # print("vals {}".format(kempt))
        # print("vals {}".format(kabs))
        # print("vals {}".format(Gb))
        # print("vals {}".format(SG))
        # print("vals {}".format(p2))

        ###########################################################################
        # Initialize GVal as a 1D array (from the first element of targets)
        target_array = np.array(targets)
        GVal = target_array            # 2D array from 3D, removed the third dimension
        # print('GVal:', GVal.shape)
        pmVal = np.zeros(GVal.shape, dtype=np.float32)
        Isc1Val = y_initIsc1[:, 0]      # Keep 2D, removed third dimension
        Isc2Val = y_initIsc2[:, 0]      # Keep 2D, removed third dimension
        IpVal = y_initIp[:, 0]          # Keep 2D, removed third dimension
        Qsto1Val = np.zeros(GVal.shape, dtype=np.float32)
        Ipb = np.zeros(GVal.shape, dtype=np.float32)
        Qsto2Val = np.zeros(GVal.shape, dtype=np.float32)
        QgutVal = np.zeros(GVal.shape, dtype=np.float32)
        RatVal = np.zeros(GVal.shape, dtype=np.float32)
        insVal = np.zeros(GVal.shape, dtype=np.float32)
        xVal = np.zeros(GVal.shape, dtype=np.float32)
        gVal = target_array            # 2D array from 3D, removed the third dimension
        y_ins = np.zeros(GVal.shape, dtype=np.float32)
        meal = np.zeros(GVal.shape, dtype=np.float32)

        # No need for np.expand_dims since everything is 1D now


        limitLoop = 43
        tau = 1
        stableEps = 100000.0

        kempt_expanded = np.tile(kd, (44 // 13 + 1, 1))[:44, :]
        p2_expanded = np.tile(kd, (44 // 13 + 1, 1))[:44, :]
        SI_expanded = np.tile(kd, (44 // 13 + 1, 1))[:44, :]

        for i in range(1, limitLoop):
            ka1 = 0.0
            
            # For 1D arrays, adjust the indexing and remove the second and third dimensions
            kd_expanded = np.tile(kd, (44 // 13 + 1, 1))[:44, :]
            # print('kd:', kd.shape)
            # print('ks:', ke)
            Ipb = (kd_expanded / ke * train_u2ss[i-1]) / kd_expanded
            # shape1 = GVal[i-1].shape

            # Update the logic to work with 1D arrays
            D = np.where((GVal[i-1] >= 60.0) & (GVal[i-1] < 119.13), 1.0, 0.0)
            E = np.where(GVal[i-1] < 60.0, 1.0, 0.0)
            
            risk = np.abs(
                10 * np.square(np.power(np.log(GVal[i - 1]), r2) - np.power(np.log(119.13), r2)) * D + 
                10 * np.power(np.power(np.log(60), r2) - np.power(np.log(119.13), r2), 2) * E
            )


            # Update all state variables to 1D
            dummyIsc1 = Isc1Val[i-1] + tau * (-kd_expanded * Isc1Val[i-1] + (train_basal[i-1] + train_ins[i-1]) / Vi)
            dummyIsc2 = Isc2Val[i-1] + tau * (kd * Isc1Val[i-1] - ka2 * Isc2Val[i-1])
            dummyIp = IpVal[i-1] + tau * (ka2 * Isc2Val[i-1] - ke * IpVal[i-1])
            dummyQsto1 = Qsto1Val[i-1] + tau * (-kempt_expanded * Qsto1Val[i-1] + train_meal[i-1])
            dummyQsto2 = Qsto2Val[i-1] + tau * (kempt * Qsto1Val[i-1] - kempt * Qsto2Val[i-1])
            dummyQgut = QgutVal[i-1] + tau * (kempt * Qsto2Val[i-1] - kabs * QgutVal[i-1])
            
            RatVal = f * kabs * QgutVal[i-1]
            dummyXVal = xVal[i-1] + tau * (-p2_expanded * xVal[i-1] - SI_expanded * (IpVal[i-1] - Ipb))
            dummygVal = gVal[i-1] + tau * (-(SG + risk * xVal[i-1]) * gVal[i-1] + SG * Gb + RatVal / Vg)

            dummyG1 = GVal[i-1] + tau * (-(1 / alpha) * (GVal[i-1] - gVal[i-1]))
            diffDummy = dummyG1 - GVal[i-1]
            sumDiff = np.sum(np.square(diffDummy)) / 256 - stableEps
            dummyG1 = np.where(sumDiff > 0.0, GVal[i-1], dummyG1)

            # Append new values to the corresponding arrays
            Isc1Val = np.append(Isc1Val, dummyIsc1)
            Isc2Val = np.append(Isc2Val, dummyIsc2)
            IpVal = np.append(IpVal, dummyIp)
            Qsto1Val = np.append(Qsto1Val, dummyQsto1)
            Qsto2Val = np.append(Qsto2Val, dummyQsto2)
            QgutVal = np.append(QgutVal, dummyQgut)
            gVal = np.append(gVal, dummygVal)
            xVal = np.append(xVal, dummyXVal)
            GVal = np.append(GVal, dummyG1)

        # Calculate error over the loop
        err = np.sqrt(np.mean(np.square(targets[limitLoop] - GVal)))
        print('err:', err)
            # err = np.sqrt(np.mean(np.square(targets[limitLoop] - GVal)))

            # # Break the loop and return the error if it's below the threshold
            # if err < 3.2:
            #     print('err:', err)
            #     return err

        return err




    


    def adamw(self, beta1=0.9, beta2=0.999, epsilon=1e-8, L2_reg=1e-4):
        """
        Updates the LTC's parameters using the AdamW optimization algorithm.
        """
        # Initialize LTC optimizer parameters if not done yet
        if not self.ltc_m_params and self.ltc_cell.is_built:
            ltc_params = self.ltc_cell.get_parameters()
            self.ltc_m_params = [np.zeros_like(param, dtype=np.float64) for param in ltc_params]
            self.ltc_v_params = [np.zeros_like(param, dtype=np.float64) for param in ltc_params]
        
        # AdamW update for LTC parameters (only if enabled)
        if self.train_ltc_params and self.ltc_cell.is_built and self.ltc_gradients:
            ltc_params = self.ltc_cell.get_parameters()
            updated_params = []
            
            for i, (param, grad, m, v) in enumerate(zip(ltc_params, self.ltc_gradients, 
                                                       self.ltc_m_params, self.ltc_v_params)):
                # AdamW update with reduced learning rate for stability
                ltc_lr = self.learning_rate * 0.1  # Use 10% of normal learning rate for LTC
                m = beta1 * m + (1 - beta1) * grad
                v = beta2 * v + (1 - beta2) * np.square(grad)
                m_hat = m / (1 - beta1)
                v_hat = v / (1 - beta2)
                param_updated = param - ltc_lr * (m_hat / (np.sqrt(v_hat) + epsilon) + L2_reg * param)
                
                # Update stored momentum and velocity
                self.ltc_m_params[i] = m
                self.ltc_v_params[i] = v
                updated_params.append(param_updated)
            
            # Set updated parameters back to LTC cell
            self.ltc_cell.set_parameters(updated_params)
        
        # Enable LTC parameter training after some iterations
        self.iteration_count += 1
        if self.iteration_count > 50 and not self.train_ltc_params:
            print("Enabling LTC parameter training...")
            self.train_ltc_params = True
        
        # AdamW update for output layer weights (Wy)
        self.mWy = beta1 * self.mWy + (1 - beta1) * self.dWy
        self.vWy = beta2 * self.vWy + (1 - beta2) * np.square(self.dWy)
        m_hat = self.mWy / (1 - beta1)
        v_hat = self.vWy / (1 - beta2)
        self.Wy -= self.learning_rate * (m_hat / (np.sqrt(v_hat) + epsilon) + L2_reg * self.Wy)

        # AdamW update for output layer bias (by)
        self.mby = beta1 * self.mby + (1 - beta1) * self.dby
        self.vby = beta2 * self.vby + (1 - beta2) * np.square(self.dby)
        m_hat = self.mby / (1 - beta1)
        v_hat = self.vby / (1 - beta2)
        self.by -= self.learning_rate * (m_hat / (np.sqrt(v_hat) + epsilon) + L2_reg * self.by)

    def train(self, data_generator,iterations):
        """
        Train the LTC on a dataset using backpropagation through time.

        Args:
            data_generator: An instance of DataGenerator containing the training data.

        Returns:
            None
        """
        iter_num = 0
        # stopping criterion for training
        threshold = 50
    
        smooth_loss = -np.log(1.0 / data_generator.vocab_size) * self.sequence_length  # initialize loss

        har_data_instance = HarData()  # Create an instance of HarData

        # Convert 3D array self.y_initIsc1 to 2D by taking the first slice along the first axis
        self.y_initIsc1 = har_data_instance.train_initIsc1[0, :, :]
        # print('y_initIsc1 (converted to 2D):', self.y_initIsc1.shape)

        # Convert 3D array self.y_initIsc2 to 2D by taking the first slice along the first axis
        self.y_initIsc2 = har_data_instance.train_initIsc2[0, :, :]
        # print('y_initIsc2 (converted to 2D):', self.y_initIsc2.shape)

        # Convert 3D array self.y_initIp to 2D by taking the first slice along the first axis
        self.y_initIp = har_data_instance.train_initIp[0, :, :]
        # print('y_initIp (converted to 2D):', self.y_initIp.shape)

        # Convert 3D array self.train_u2ss to 2D by taking the first slice along the first axis
        self.train_u2ss = har_data_instance.train_u2ss[0, :, :]
        # print('train_u2ss (converted to 2D):', self.train_u2ss.shape)

        # Convert 3D array self.train_basal to 2D by taking the first slice along the first axis
        self.train_basal = har_data_instance.train_basal[0, :, :]
        # print('train_basal (converted to 2D):', self.train_basal.shape)

        # Convert 3D array self.train_ins to 2D by taking the first slice along the first axis
        self.train_ins = har_data_instance.train_ins[0, :, :]
        # print('train_ins (converted to 2D):', self.train_ins.shape)

        # Convert 3D array self.train_meal to 2D by taking the first slice along the first axis
        self.train_meal = har_data_instance.train_meal[0, :, :]
        # print('train_meal (converted to 2D):', self.train_meal.shape)


        early_stop_patience = 3  # The number of consecutive iterations with small loss change
        consecutive_small_loss_change = 0  # Counter for consecutive small loss changes
        threshold = 0.10  # 10% threshold for loss change
        prev_loss = None  # To store the previous loss value
        
        while (iter_num < iterations):
            # initialize hidden state at the beginning of each sequence
            if data_generator.pointer == 0:
                c_prev = np.zeros((self.hidden_size, 1))
                a_prev = np.zeros((self.hidden_size, 1))

            # get a batch of inputs and targets
            inputs, targets = data_generator.next_batch()


            # forward passy
            X, r, z, c, cc, a, y_pred = self.forward(inputs, c_prev, a_prev)
            # print('y_pred_list:', len(y_pred))
            # print('y_pred_list:', y_pred[199].shape)

            # backward pass
            self.backward(X, a_prev, c_prev, r, z, c, cc, a, y_pred, targets)

            # calculate and update loss
            loss = self.loss(y_pred, targets, self.y_initIsc1, self.y_initIsc2, self.y_initIp, self.train_u2ss, self.train_basal, self.train_ins, self.train_meal)

            # loss =  np.sqrt(np.mean(np.square(np.array(list(y_pred.values())) - np.array(targets))))



             # Early stopping logic
            # if prev_loss is not None:
            #     # Calculate the percentage change in loss
            #     loss_diff = abs(prev_loss - loss) / prev_loss

            #     if loss_diff < threshold:  # If the loss change is less than 10%
            #         consecutive_small_loss_change += 1
            #         if consecutive_small_loss_change >= early_stop_patience:
            #             print(f"Early stopping triggered at iteration {iter_num} with loss: {loss}")
            #             break
            #     else:
            #         consecutive_small_loss_change = 0  # Reset the counter if the change is larger than 10%

            # prev_loss = loss  # Update the previous loss

            print('loss:', loss)
            # print('loss shape:')
            self.adamw()
            smooth_loss = smooth_loss * 0.999 + loss * 0.001
            
            # Print additional training info every 10 iterations
            if iter_num % 10 == 0:
                ltc_status = "enabled" if self.train_ltc_params else "disabled"
                print(f"Iteration {iter_num}: smooth_loss={smooth_loss:.4f}, LTC training={ltc_status}")
            # update previous hidden state for the next batch
            a_prev = a[self.sequence_length - 1]
            c_prev = c[self.sequence_length - 1]
#             if iter_num == 5900 or iter_num == 30000:
#                         self.learning_rate *= 0.1
            # print progress every 100 iterations
            if iter_num % 100 == 0:
#                 self.learning_rate *= 0.99
                sample_idx = self.sample(c_prev, a_prev, inputs[0, :], 200)
                print(''.join(data_generator.idx_to_char[idx] for idx in sample_idx))
                print("\n\niter :%d, loss:%f" % (iter_num, smooth_loss))
            iter_num += 1
            
        return smooth_loss, y_pred

    def sample(self, c_prev, a_prev, seed_idx, n):
        """
        Sample a sequence of integers from the LTC model.

        Args:
            c_prev (numpy.ndarray): Previous cell state, a numpy array of shape (hidden_size, 1) - unused for LTC.
            a_prev (numpy.ndarray): Previous hidden state, a numpy array of shape (hidden_size, 1).
            seed_idx (numpy.ndarray): Seed letter from the first time step, a numpy array of shape (vocab_size, 1).
            n (int): Number of characters to generate.

        Returns:
            list: A list of integers representing the generated sequence.

        """
        # initialize input and seed_idx
        x = np.zeros((self.vocab_size, 1))
        # convert one-hot encoding to integer index
        seed_idx = np.argmax(seed_idx, axis=-1)

                # set the seed letter as the input for the first time step
        x[seed_idx] = 1

        # Initialize state
        state = a_prev.flatten()
        
        # generate sequence of characters
        idxes = []
        for t in range(n):
            # Get input as 1D array
            input_t = x.flatten()  # Shape: (vocab_size,)
            
            # Forward pass through LTC cell
            ltc_output, state = self.ltc_cell.forward(input_t, state)
            
            # Reshape for compatibility
            hidden_state = ltc_output.reshape(-1, 1)  # Shape: (hidden_size, 1)
            
            # compute the output probabilities
            y = self.softmax(np.dot(self.Wy, hidden_state) + self.by)

            # sample the next character from the output probabilities
            idx = np.random.choice(range(self.vocab_size), p=y.ravel())

            # set the input for the next time step
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1

            # append the sampled character to the sequence
            idxes.append(idx)

        # return the generated sequence
        return idxes

    def predict(self, data_generator, start, n):
        """
        Generate a sequence of n characters using the trained LTC model, starting from the given start sequence.

        Args:
        - data_generator: an instance of DataGenerator
        - start: a string containing the start sequence
        - n: an integer indicating the length of the generated sequence

        Returns:
        - txt: a string containing the generated sequence
        """
        # initialize input sequence
        x = np.zeros((self.vocab_size, 1))
        chars = [ch for ch in start]
        idxes = []
        for i in range(len(chars)):
            idx = data_generator.char_to_idx[chars[i]]
            x[idx] = 1
            idxes.append(idx)
            
        # initialize hidden state
        state = np.zeros(self.hidden_size)

        # generate new sequence of characters
        for t in range(n):
            # Get input as 1D array
            input_t = x.flatten()  # Shape: (vocab_size,)
            
            # Forward pass through LTC cell
            ltc_output, state = self.ltc_cell.forward(input_t, state)
            
            # Reshape for compatibility
            hidden_state = ltc_output.reshape(-1, 1)  # Shape: (hidden_size, 1)

            # compute the output probability vector
            y_pred = self.softmax(np.dot(self.Wy, hidden_state) + self.by)
            # sample the next character from the output probabilities
            idx = np.random.choice(range(self.vocab_size), p=y_pred.ravel())
            x = np.zeros((self.vocab_size, 1))
            x[idx] = 1
            idxes.append(idx)
        txt = ''.join(data_generator.idx_to_char[i] for i in idxes)
        return txt


In [35]:
import time
losses10 = []
y_pred_list = []
cumulative_time = []

sequence_length = 200
#read text from the "input.txt" file
data_generator = DataGenerator('Virtual_570_13_glucose.txt', sequence_length)
ltc =  LTC(hidden_size=16, vocab_size=data_generator.vocab_size,sequence_length=sequence_length,learning_rate=0.001)

start_time = time.time()
smooth_loss, y_pred = ltc.train(data_generator, iterations=200)
end_time = time.time()

elapsed_time = end_time - start_time
print(f"Training took {elapsed_time:.2f} seconds")

losses10.append(smooth_loss)
y_pred_list.append(y_pred[199])
cumulative_time.append(elapsed_time)


Parsing for Patient File 13
Total number of testing sequences: 6
Validation split: 4, training split: 44
Total number of test sequences: 6
err: 5.790416705197883
loss: 5.790416705197883
Iteration 0: smooth_loss=512.4827, LTC training=disabled
42
26847
97
4 1 7.36..44529996
188.3
5
0912675.6.41
4194.6433582 .4663.776475372 404821..28970810794 .39414275. .8.820
3
144175
20419668 449599..673 891531..
5881...95039978992
12480288832 9
.3978
.24


iter :0, loss:512.482672


C:\Users\binxu4\AppData\Local\Temp\ipykernel_43756\1681122234.py:347: RuntimeWarning: divide by zero encountered in log
  10 * np.square(np.power(np.log(GVal[i - 1]), r2) - np.power(np.log(119.13), r2)) * D +
C:\Users\binxu4\AppData\Local\Temp\ipykernel_43756\1681122234.py:347: RuntimeWarning: invalid value encountered in multiply
  10 * np.square(np.power(np.log(GVal[i - 1]), r2) - np.power(np.log(119.13), r2)) * D +


err: 6.448140227747394
loss: 6.448140227747394
err: 4.922120755047841
loss: 4.922120755047841
err: 6.06200741374485
loss: 6.06200741374485
err: 4.289618449965101
loss: 4.289618449965101
err: 4.207431441965867
loss: 4.207431441965867
err: 6.7174375279676966
loss: 6.7174375279676966
err: 7.507571935295483
loss: 7.507571935295483
err: 7.306455879608195
loss: 7.306455879608195
err: 5.246013835001276
loss: 5.246013835001276
err: 5.878241149209895
loss: 5.878241149209895
Iteration 10: smooth_loss=507.4392, LTC training=disabled
err: 5.772906224856917
loss: 5.772906224856917
err: 4.217731167149835
loss: 4.217731167149835
err: 7.375691591049683
loss: 7.375691591049683
err: 6.68382777448733
loss: 6.68382777448733
err: 4.070149346458842
loss: 4.070149346458842
err: 4.672320463366859
loss: 4.672320463366859
err: 4.174400535970493
loss: 4.174400535970493
err: 7.075157329676098
loss: 7.075157329676098
err: 5.714632516040592
loss: 5.714632516040592
err: 6.629199121442615
loss: 6.629199121442615
Iter

In [36]:
print('smooth_loss:', losses10)
print('y_pred_list:', y_pred_list)
print('cumulative_time:', cumulative_time)

total_time = 0  # Initialize the sum

# Loop to add each value
for time_value in cumulative_time:
    total_time += time_value

print(f"Total time: {total_time:.5f} seconds")

smooth_loss: [420.99746389791653]
y_pred_list: [array([[0.06769158],
       [0.07039391],
       [0.10533036],
       [0.06189423],
       [0.05018076],
       [0.11799774],
       [0.06375914],
       [0.06083138],
       [0.0533902 ],
       [0.05543627],
       [0.0601749 ],
       [0.10306642],
       [0.1298531 ]])]
cumulative_time: [19.57876491546631]
Total time: 19.57876 seconds


In [37]:
print('y_pred_list:', y_pred_list[0].shape)

y_pred_list: (13, 1)


In [38]:
ltc.predict(data_generator, "9", 1000)

'95237731 6.0897.9141932 40341\n70 6998 9 7.91685 1 908 28238 1.4.0491.5308  .6\n.9.92.78.40445434 23 7 483 92148.112 9\n30658940.84..124.0 02.44\n 88.837 4.1 9. .3171\n93 66.294.1 7\n500654\n8 8215.8 3.33. 2 3 2.25\n 38.7.58 6 2.35. 92475.3479 . 7. .17 080497.248120.03.6  872452 72021  \n .\n3894904 4421590278 .8 1.0433 . 59 .45\n.\n667  55349 20 3 82 239\n99313\n \n2.49 2594 ..924\n69 361..\n7 2 2144201574816. 06513157..4 0  7.6\n2 1\n 283128.482 2.15 ..748442836212 556.12222458341 14887 62 4\n78356\n89211.82.4.2.45.2 ..3 7 \n28\n12592.08\n20. .\n71878.50. 999427\n2.5999509.66435 \n2.6146.105 122.145.1.99 716.902805225 98 229 5 26101 8.8\n06157127870   .119.4.0. 1.46614126595  .628381.6 8.9  72584048.918937222.651 \n\n0  94.14928\n\n9080\n5.218\n..681483\n 654 5578.49 9.14394\n62448.. 406 536 .66361163.819 152194 7.02 7\n0673.  .96 0.677 3.39823 \n2..198 8 5 222.72 2 4 36.788.20167.12.\n417 04488562 3.0 49941 7986009.2..48143\n.116 44221.73. 927464097046..50 7 7.973446  2 49.1413.204

<a id="4"></a>
<h1 style='background:#FF9F00;border:0; color:black;
    box-shadow: 10px 10px 5px 0px rgba(0,0,0,0.75);
    transform: rotateX(10deg);
    '><center style='color: #3E3D53;'>Thank you</center></h1>

# Thank you

**Thank you for going through this notebook**

**If you have any suggestions please let me know**
